In [1]:
import pandas as pd

base = "https://minio.lab.sspcloud.fr/mangane/Projet_PDS_2026/processed/"
fs = pd.read_csv(base + "finess_sygen_final.csv", dtype=str)
insee = pd.read_csv(base + "insee_communes_final.csv", dtype=str)

In [2]:
def harmoniser(code):
    if pd.isna(code):
        return code
    if "75101" <= code <= "75120":
        return "75056"   # Paris
    if "69381" <= code <= "69389":
        return "69123"   # Lyon
    if "13201" <= code <= "13216":
        return "13055"   # Marseille
    return code

fs2 = fs.dropna(subset=["CODGEO"]).copy()
fs2["CODGEO_ORIG"] = fs2["CODGEO"]
fs2["CODGEO"] = fs2["CODGEO"].map(harmoniser)
fs3 = fs2[fs2["ETAT"] == "A"].copy()

print(len(fs), len(fs2), len(fs3))

3988 3940 3921


In [3]:
cols_num = ["LIT_MED", "LIT_CHI", "LIT_OBS", "LIT_MCO", "LIT_SMR", "LIT_USLD",
            "NB_SCAN", "NB_IRM", "NB_MAMM"]
for c in cols_num:
    fs3[c] = pd.to_numeric(fs3[c], errors="coerce")

hopitaux = fs3.groupby("CODGEO")[cols_num].sum(min_count=1)
hopitaux["nb_etablissements"] = fs3.groupby("CODGEO")["FI"].count()
hopitaux = hopitaux.reset_index()

print(hopitaux.shape)
print(hopitaux["nb_etablissements"].sum())
print(hopitaux.head())

(1714, 11)
3921
  CODGEO  LIT_MED  LIT_CHI  LIT_OBS  LIT_MCO  LIT_SMR  LIT_USLD  NB_SCAN  \
0  01004     31.0     27.0     15.0     73.0      NaN       NaN      NaN   
1  01033      NaN      NaN      NaN      NaN      NaN       NaN      NaN   
2  01034     56.0     19.0      8.0     83.0     40.0       NaN      1.0   
3  01053     26.0     44.0      NaN     70.0    156.0      11.0      0.0   
4  01082      NaN      NaN      NaN      NaN     70.0       NaN      NaN   

   NB_IRM  NB_MAMM  nb_etablissements  
0     NaN      NaN                  3  
1     NaN      NaN                  1  
2     0.0      1.0                  2  
3     0.0      0.0                  5  
4     NaN      NaN                  1  


## Partie 2 : fusion avec les données INSEE

In [4]:
# communes avec établissement absentes de l'INSEE 
print(sorted(set(hopitaux["CODGEO"]) - set(insee["CODGEO"])))

fusion = insee.merge(hopitaux, on="CODGEO", how="left", validate="one_to_one")
fusion["nb_etablissements"] = fusion["nb_etablissements"].fillna(0)

print(fusion.shape)
print("communes avec au moins 1 établissement :", (fusion["nb_etablissements"] > 0).sum())
print(fusion[fusion["CODGEO"] == "01004"].T)

['97123', '97127', '97502', '97606', '97611', '97615']
(34858, 23)
communes avec au moins 1 établissement : 1708
                                                 2
CODGEO                                       01004
LIBGEO                           Ambérieu-en-Bugey
population                                 15554.0
population_65_plus                2809.27789483767
part_65_plus                     0.180614497546462
part_65_plus_pct                  18.0614497546462
DENS                                           2.0
LIBDENS                       Urbain intermédiaire
DENS7                                          2.0
LIBDENS7            Centres urbains intermédiaires
MED21                                        21660
TP6021                                        17.0
RD21                                           3.1
LIT_MED                                       31.0
LIT_CHI                                       27.0
LIT_OBS                                       15.0
LIT_MCO             

In [5]:
print("établissements dans la fusion :", fusion["nb_etablissements"].sum(), "sur", len(fs3))
print("codes 97 dans l'INSEE :", insee["CODGEO"].str.startswith("97").sum())
print(fs3[fs3["CODGEO"].isin(['97123', '97127', '97502', '97606', '97611', '97615'])][["CODGEO", "NOM", "CATEGORIE"]])


établissements dans la fusion : 3906.0 sur 3921
codes 97 dans l'INSEE : 112
     CODGEO                                                NOM CATEGORIE
3853  97123  CENTRE HOSPITALIER IRENEE DE BRUYN EX HÔPITAL ...       106
3855  97127          CENTRE HOSPITALIER LOUIS CONSTANT FLEMING       355
3868  97127                      ANTENNE AUDRA DE SAINT MARTIN       146
3875  97127                                  HAD  ILES DU NORD       127
3879  97123  CH LOUIS CONSTANT FLEMING - ANTENNE SMUR DE SA...       355
3880  97123            CLINIQUE DE CHOISY HAD SAINT BARTHELEMY       127
3881  97127                                    CLINIQUE WATAKI       109
3980  97502                 CENTRE HOSPITALIER  FRANCOIS DUNAN       355
3981  97611                      CENTRE HOSPITALIER DE MAYOTTE       355
3982  97611               UNITE AUTODIALYSE ASSISTEE MAMOUDZOU       141
3983  97611                                         UDM KAWENI       141
3984  97606                   UNITE D'AUTODIALYS

In [6]:
outre_mer = insee[insee["CODGEO"].str.startswith("97")]
print(len(outre_mer))
print(outre_mer["CODGEO"].str[:3].value_counts().sort_index())

112
CODGEO
971    32
972    34
973    22
974    24
Name: count, dtype: int64


**Périmètre.** 15 établissements actifs (0,4 % du total), situés à Saint-Martin, Saint-Barthélemy, Saint-Pierre-et-Miquelon et Mayotte, sont exclus : ces territoires sont absents des tables INSEE de population, de revenus et de densité, ce qui empêche de calculer un indicateur pour eux. L'analyse couvre la France métropolitaine et les quatre départements d'outre-mer historiques (Guadeloupe, Martinique, Guyane, La Réunion)

In [7]:
hors_insee = hopitaux[~hopitaux["CODGEO"].isin(insee["CODGEO"])]
print(len(hors_insee), "communes hors table INSEE,",
      int(hors_insee["nb_etablissements"].sum()), "établissements exclus")

6 communes hors table INSEE, 15 établissements exclus


In [8]:
fusion["nb_etablissements"] = fusion["nb_etablissements"].fillna(0).astype(int)

In [9]:
print(fusion["nb_etablissements"].dtype)
print(fusion["nb_etablissements"].isna().sum())
print((fusion["nb_etablissements"] == 0).sum())

int64
0
33150


In [10]:
fusion.head()

,CODGEO,LIBGEO,population,population_65_plus,part_65_plus,part_65_plus_pct,DENS,LIBDENS,DENS7,LIBDENS7,...,LIT_MED,LIT_CHI,LIT_OBS,LIT_MCO,LIT_SMR,LIT_USLD,NB_SCAN,NB_IRM,NB_MAMM,nb_etablissements
0,01001,L'Abergement-Clémenciat,859.0,172.0,0.20023282887077998,20.023282887077997,3.0,Rural,6.0,Rural à habitat dispersé,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
1,01002,L'Abergement-de-Varey,273.0,50.0,0.18315018315018314,18.315018315018314,3.0,Rural,6.0,Rural à habitat dispersé,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
2,01004,Ambérieu-en-Bugey,15554.0,2809.27789483767,0.180614497546462,18.0614497546462,2.0,Urbain intermédiaire,2.0,Centres urbains intermédiaires,...,31.0,27.0,15.0,73.0,NaN,NaN,NaN,NaN,NaN,3
3,01005,Ambérieux-en-Dombes,1916.9999999999982,325.60178927474755,0.16984965533372345,16.984965533372346,3.0,Rural,5.0,Bourgs ruraux,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
4,01006,Ambléon,114.0,32.0,0.2807017543859649,28.07017543859649,3.0,Rural,6.0,Rural à habitat dispersé,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
